# AI-Generated Text Detection — MALTO Hackathon

**Task:** Multi-class classification (6 classes)
- `0` Human
- `1` DeepSeek
- `2` Grok
- `3` Claude
- `4` Gemini
- `5` ChatGPT

**Metric:** Macro F1 Score

**Approach:** Ensemble of 3 TF-IDF + Logistic Regression models (word n-grams, char n-grams, handcrafted features) with meta-learner stacking.

## 1. Imports

In [1]:
import re
from typing import Any

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

## 2. Load Data

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
train.head()

Train shape: (2400, 2)
Test shape:  (600, 2)


,TEXT,LABEL
0,"Dear state senator, There should be a change i...",0
1,A star's life cycle begins in a nebula and pro...,1
2,Limiting the usage of has a variety advantages...,0
3,Biodiversity loss accelerates at unprecedented...,3
4,The morning had been cold. The hours dawn stil...,0


## 3. Exploratory Data Analysis

In [3]:
train.info()
print()
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TEXT    2400 non-null   object
 1   LABEL   2400 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 37.6+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  600 non-null    int64 
 1   TEXT        600 non-null    object
dtypes: int64(1), object(1)
memory usage: 9.5+ KB


In [4]:
print("Null values in train:")
print(train.isnull().sum())
print("\nNull values in test:")
print(test.isnull().sum())

Null values in train:
TEXT     0
LABEL    0
dtype: int64

Null values in test:
Unnamed: 0    0
TEXT          0
dtype: int64


In [5]:
label_names = {0: "Human", 1: "DeepSeek", 2: "Grok", 3: "Claude", 4: "Gemini", 5: "ChatGPT"}

print("Label distribution (train):")
print(train["LABEL"].value_counts().sort_index().rename(label_names))

Label distribution (train):
LABEL
Human       1520
DeepSeek      80
Grok         160
Claude        80
Gemini       240
ChatGPT      320
Name: count, dtype: int64


In [6]:
# Text length stats by class
train["char_len"] = train["TEXT"].str.len()
train["word_count"] = train["TEXT"].str.split().str.len()

stats = train.groupby("LABEL")[["char_len", "word_count"]].agg(["mean", "median", "min", "max"]).round(1)
stats.index = stats.index.map(label_names)
stats

char_len                     word_count                  
             mean  median   min   max       mean median  min   max
LABEL                                                             
Human      2510.6  2537.5   403  9685      450.9  452.5   74  1910
DeepSeek    560.0   172.0    91  1722       83.1   23.5   13   277
Grok        414.5   232.0   117  1388       55.6   33.0   15   197
Claude     3358.7  3256.0  1898  5221      375.6  359.5  199   632
Gemini     2818.7  2855.5  1102  5514      419.9  424.5  161   800
ChatGPT    1485.2  1321.5   952  3415      179.4  153.0  105   473

In [7]:
# Vocabulary richness: unique words / total words — higher = less repetition
train["vocab_richness"] = train["TEXT"].apply(
    lambda t: len(set(t.lower().split())) / max(len(t.split()), 1)
)

richness = train.groupby("LABEL")["vocab_richness"].mean().round(3)
richness.index = richness.index.map(label_names)
print("Mean vocabulary richness by class (lower = more repetitive):")
print(richness.sort_values())

Mean vocabulary richness by class (lower = more repetitive):
LABEL
Human       0.574
Gemini      0.616
Claude      0.650
ChatGPT     0.743
DeepSeek    0.875
Grok        0.913
Name: vocab_richness, dtype: float64


In [8]:
# Sample one text per class
print("=" * 70)
for label, name in label_names.items():
    sample = train[train["LABEL"] == label]["TEXT"].iloc[0]
    print(f"\n[{name}]  chars={len(sample)}  words={len(sample.split())}")
    print("-" * 70)
    print(sample[:400] + ("..." if len(sample) > 400 else ""))
    print("=" * 70)


[Human]  chars=2980  words=520
----------------------------------------------------------------------
Dear state senator, There should be a change in the Electoral College. It should be changed to electing presidents by popular vote. It is our right to vote for someone who would actually make changes in our society and make our ilves different. The fact that we have to vote electors for those electors to choose our president, it seems unfair. Also, the purpose of voting for president is for everyb...

[DeepSeek]  chars=146  words=26
----------------------------------------------------------------------
A star's life cycle begins in a nebula and progresses through stages determined by its mass, ending as a white dwarf, neutron star, or black hole.

[Grok]  chars=147  words=18
----------------------------------------------------------------------
The Industrial Revolution, which began in the eighteenth century, transformed economies through the widespread adoption of steam-powered machi

## 4. Feature Engineering

### 4a. Handcrafted Features

Statistical text features that capture stylistic differences: length, sentence structure, vocabulary richness, punctuation patterns, contraction rate, formality markers.

In [9]:
def extract_handcrafted_features(texts: np.ndarray) -> np.ndarray:
    rows: list[list[Any]] = []
    for text in texts:
        words = text.split()
        sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

        char_len = len(text)
        word_count = len(words)
        num_sentences = max(len(sentences), 1)

        sent_word_lens = [len(s.split()) for s in sentences] if sentences else [0]
        avg_sent_len = np.mean(sent_word_lens)
        sent_len_var = np.var(sent_word_lens)
        avg_word_len = np.mean([len(w) for w in words]) if words else 0

        unique_words = len(set(w.lower() for w in words))
        vocab_richness = unique_words / max(word_count, 1)

        upper_ratio = sum(1 for c in text if c.isupper()) / max(char_len, 1)
        allcaps_words = sum(1 for w in words if w.isupper() and len(w) > 2)
        allcaps_ratio = allcaps_words / max(word_count, 1)

        punct_rate = sum(1 for c in text if c in ".,!?;:") / max(char_len, 1)
        comma_rate = text.count(",") / max(char_len, 1)
        repeated_chars = len(re.findall(r"(.)\1{2,}", text)) / max(char_len, 1)
        newline_rate = text.count("\n") / max(char_len, 1)

        has_bullets = 1 if re.search(r"^\s*[-*]\s", text, re.MULTILINE) else 0
        has_numbered = 1 if re.search(r"^\s*\d+[.)]\s", text, re.MULTILINE) else 0

        is_short = 1 if char_len < 300 else 0
        is_single_sentence = 1 if num_sentences <= 1 else 0
        is_long = 1 if char_len > 2000 else 0

        tl = text.lower()
        contractions = sum(
            1
            for p in [
                "i'm", "it's", "don't", "can't", "won't", "i've",
                "you're", "that's", "isn't", "didn't", "couldn't",
                "they're", "we're",
            ]
            if p in tl
        )
        contraction_rate = contractions / max(word_count, 1)

        formal_transitions = sum(
            1
            for p in [
                "moreover,", "furthermore,", "in conclusion,",
                "in summary,", "additionally,", "therefore,", "however,",
            ]
            if p in tl
        )

        rows.append([
            char_len, word_count, num_sentences, avg_sent_len, sent_len_var,
            avg_word_len, vocab_richness, upper_ratio, allcaps_ratio,
            punct_rate, comma_rate, repeated_chars, newline_rate,
            has_bullets, has_numbered, is_short, is_single_sentence,
            contraction_rate, formal_transitions, is_long,
        ])

    return np.array(rows, dtype=np.float32)

### 4b. TF-IDF + Handcrafted Feature Matrix Builder

In [10]:
def build_sparse_features(
    x_train_raw: np.ndarray,
    x_test_raw: np.ndarray,
    word_ngram: tuple[int, int],
    char_ngram: tuple[int, int],
    word_max_features: int,
    char_max_features: int,
    word_min_df: int,
    char_min_df: int,
    include_hcf: bool,
) -> tuple[csr_matrix, csr_matrix]:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngram,
        max_features=word_max_features,
        sublinear_tf=True,
        min_df=word_min_df,
        strip_accents="unicode",
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=char_ngram,
        max_features=char_max_features,
        sublinear_tf=True,
        min_df=char_min_df,
    )

    x_word_train = csr_matrix(word_vectorizer.fit_transform(x_train_raw))
    x_word_test = csr_matrix(word_vectorizer.transform(x_test_raw))
    x_char_train = csr_matrix(char_vectorizer.fit_transform(x_train_raw))
    x_char_test = csr_matrix(char_vectorizer.transform(x_test_raw))

    train_parts = [x_word_train, x_char_train]
    test_parts = [x_word_test, x_char_test]

    if include_hcf:
        hcf_train = extract_handcrafted_features(x_train_raw)
        hcf_test = extract_handcrafted_features(x_test_raw)
        scaler = StandardScaler()
        hcf_train_scaled = scaler.fit_transform(hcf_train)
        hcf_test_scaled = scaler.transform(hcf_test)
        train_parts.append(csr_matrix(hcf_train_scaled))
        test_parts.append(csr_matrix(hcf_test_scaled))

    x_train = csr_matrix(hstack(train_parts))
    x_test = csr_matrix(hstack(test_parts))
    return x_train, x_test

## 5. Cross-Validated OOF Training

Each base model uses 5-fold stratified CV to produce out-of-fold (OOF) predictions for the meta-learner.

In [11]:
def get_oof_and_test_proba(
    x_train: csr_matrix,
    y_train: np.ndarray,
    x_test: csr_matrix,
    model_params: dict[str, Any],
    n_splits: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    classes = np.unique(y_train)
    n_classes = len(classes)

    oof_proba = np.zeros((x_train.shape[0], n_classes), dtype=np.float64)
    test_proba_folds = np.zeros((x_test.shape[0], n_classes), dtype=np.float64)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for fold, (fit_idx, val_idx) in enumerate(skf.split(x_train, y_train)):
        model = LogisticRegression(**model_params)
        model.fit(x_train[fit_idx], y_train[fit_idx])
        oof_proba[val_idx] = model.predict_proba(x_train[val_idx])
        test_proba_folds += model.predict_proba(x_test)
        print(f"  Fold {fold + 1}/{n_splits} done")

    test_proba = test_proba_folds / n_splits
    return oof_proba, test_proba

## 6. Train Base Models

Three models with different TF-IDF configurations capture complementary signals.

In [12]:
x_train_raw = train["TEXT"].to_numpy()
y_train = train["LABEL"].to_numpy()
x_test_raw = test["TEXT"].to_numpy()
test_ids = test.iloc[:, 0].to_numpy()
classes = np.unique(y_train)

In [13]:
print("=== Model A: word (1-3) + char (2-5) + handcrafted features ===")
x_train_a, x_test_a = build_sparse_features(
    x_train_raw, x_test_raw,
    word_ngram=(1, 3), char_ngram=(2, 5),
    word_max_features=50000, char_max_features=50000,
    word_min_df=2, char_min_df=3,
    include_hcf=True,
)
oof_a, test_a = get_oof_and_test_proba(
    x_train_a, y_train, x_test_a,
    {"class_weight": "balanced", "solver": "lbfgs", "C": 100.0, "max_iter": 1200},
)
oof_a_preds = classes[np.argmax(oof_a, axis=1)]
print(f"Model A OOF Macro F1: {f1_score(y_train, oof_a_preds, average='macro'):.4f}")

=== Model A: word (1-3) + char (2-5) + handcrafted features ===
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done
Model A OOF Macro F1: 0.9192


In [14]:
print("=== Model B: word (1-2) + char (2-6) ===")
x_train_b, x_test_b = build_sparse_features(
    x_train_raw, x_test_raw,
    word_ngram=(1, 2), char_ngram=(2, 6),
    word_max_features=60000, char_max_features=80000,
    word_min_df=2, char_min_df=2,
    include_hcf=False,
)
oof_b, test_b = get_oof_and_test_proba(
    x_train_b, y_train, x_test_b,
    {"class_weight": "balanced", "solver": "lbfgs", "C": 60.0, "max_iter": 1200},
)
oof_b_preds = classes[np.argmax(oof_b, axis=1)]
print(f"Model B OOF Macro F1: {f1_score(y_train, oof_b_preds, average='macro'):.4f}")

=== Model B: word (1-2) + char (2-6) ===
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done
Model B OOF Macro F1: 0.8922


In [15]:
print("=== Model C: word (1-3) + char (3-5) + handcrafted features ===")
x_train_c, x_test_c = build_sparse_features(
    x_train_raw, x_test_raw,
    word_ngram=(1, 3), char_ngram=(3, 5),
    word_max_features=45000, char_max_features=70000,
    word_min_df=1, char_min_df=2,
    include_hcf=True,
)
oof_c, test_c = get_oof_and_test_proba(
    x_train_c, y_train, x_test_c,
    {"class_weight": None, "solver": "lbfgs", "C": 30.0, "max_iter": 1200},
)
oof_c_preds = classes[np.argmax(oof_c, axis=1)]
print(f"Model C OOF Macro F1: {f1_score(y_train, oof_c_preds, average='macro'):.4f}")

=== Model C: word (1-3) + char (3-5) + handcrafted features ===
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done
Model C OOF Macro F1: 0.9097


## 7. Meta-Learner Stacking

A Logistic Regression meta-learner is trained on the OOF probability outputs of the three base models. Final prediction blends the meta-learner (75%) with a soft-vote ensemble (25%).

In [16]:
# Soft-vote baseline
base_test_proba = 0.45 * test_a + 0.35 * test_b + 0.20 * test_c

# Meta-learner stacking
x_meta_train = np.hstack([oof_a, oof_b, oof_c])
x_meta_test = np.hstack([test_a, test_b, test_c])

meta_model = LogisticRegression(
    class_weight="balanced", solver="lbfgs", C=6.0, max_iter=2000
)
meta_model.fit(x_meta_train, y_train)
meta_test_proba = meta_model.predict_proba(x_meta_test)

# OOF meta score
meta_oof_proba = meta_model.predict_proba(x_meta_train)
meta_oof_preds = classes[np.argmax(meta_oof_proba, axis=1)]
print(f"Meta-learner OOF Macro F1: {f1_score(y_train, meta_oof_preds, average='macro'):.4f}")

Meta-learner OOF Macro F1: 0.9370


In [17]:
# Blend meta + base predictions
test_proba = 0.75 * meta_test_proba + 0.25 * base_test_proba
test_predictions = classes[np.argmax(test_proba, axis=1)]

print("Predicted label distribution:")
unique, counts = np.unique(test_predictions, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u} ({label_names[u]}): {c}")

Predicted label distribution:
  0 (Human): 380
  1 (DeepSeek): 22
  2 (Grok): 38
  3 (Claude): 20
  4 (Gemini): 60
  5 (ChatGPT): 80


## 8. Generate Submission

In [18]:
submission = pd.DataFrame({"ID": test_ids, "LABEL": test_predictions})
submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv with {len(submission)} rows")
submission.head(10)

Saved submission.csv with 600 rows


,ID,LABEL
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,5
6,6,0
7,7,0
8,8,5
9,9,3
